In [3]:
import base64
from langchain_deepseek import ChatDeepSeek
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
class LLM:
    def __init__(self):
        load_dotenv(".env.local",override=True)
        self.api_key= os.getenv("DEEPSEEK_API_KEY").strip()
        self.model= os.getenv("MODEL").strip()
        self.url= os.getenv("DEEPSEEK_BASE_URL").strip()
        try:
            self.thinking_timeout= float(os.getenv("TINKING_TIMEOUT",30).strip())
            self.num_retries= int(os.getenv("RETRY_NUM",2).strip())
            self.temperature= int(os.getenv("TEMPERATURE",1).strip())
            self.stream= True if os.getenv("STREAM").strip() == 'True' else False
            self.max_token= int(os.getenv("MAX_TOKEN").strip()) if os.getenv("MAX_TOKEN").strip() else None
        except Exception as e:
            raise Exception(f'配置文件传入非法参数。错误信息：{e}')
        self.set_llm()
    def set_llm(self):
        self.llm = ChatDeepSeek(
            model= self.model,
            api_key= self.api_key,
            streaming= self.stream,
            api_base= self.url,
            temperature=self.temperature,
            request_timeout= self.thinking_timeout,
            max_tokens= self.max_token,
            max_retries= self.num_retries,
            # model_kwargs=   {'tools':[]}##用来存放一些langchain没有列出但模型本身支持的，比如tools
            # extra_body= {}  ##基于openai个性化字段 比如thinking
            # configurable_fields= ('model','temperature') ## 用来允许 config中的configurable 覆盖
        )

def encode_image(img_path, img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode("utf-8")}"

# 图像路径
img_path = "1.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)
model = LLM().llm
response = model.invoke(
    [
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    "base64": base64_image,
                    'mime_type':'image/png',
                }
            ]
        )
    ]
)
print(response.content)

BadRequestError: Error code: 400 - {'error': {'message': 'Failed to deserialize the JSON body into the target type: messages[0]: unknown variant `image_url`, expected `text` at line 1 column 653868', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

可以看到deepseek不支持多模态中的图片，实际查了下好像api目前只支持文本

In [4]:
import base64
from langchain_deepseek import ChatDeepSeek
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
class LLM:
    def __init__(self):
        load_dotenv(".env.local",override=True)
        self.api_key= os.getenv("DEEPSEEK_API_KEY").strip()
        self.model= os.getenv("MODEL").strip()
        self.url= os.getenv("DEEPSEEK_BASE_URL").strip()
        try:
            self.thinking_timeout= float(os.getenv("TINKING_TIMEOUT",30).strip())
            self.num_retries= int(os.getenv("RETRY_NUM",2).strip())
            self.temperature= int(os.getenv("TEMPERATURE",1).strip())
            self.stream= True if os.getenv("STREAM").strip() == 'True' else False
            self.max_token= int(os.getenv("MAX_TOKEN").strip()) if os.getenv("MAX_TOKEN").strip() else None
        except Exception as e:
            raise Exception(f'配置文件传入非法参数。错误信息：{e}')
        self.set_llm()
    def set_llm(self):
        self.llm = ChatDeepSeek(
            model= self.model,
            api_key= self.api_key,
            streaming= self.stream,
            api_base= self.url,
            temperature=self.temperature,
            request_timeout= self.thinking_timeout,
            max_tokens= self.max_token,
            max_retries= self.num_retries,
            extra_body={"thinking":{"type":"enabled"}}
            # model_kwargs=   {'tools':[]}##用来存放一些langchain没有列出但模型本身支持的，比如tools
            # configurable_fields= ('model','temperature') ## 用来允许 config中的configurable 覆盖
        )

model = LLM().llm
response = model.invoke(
    [
        HumanMessage(
            content="你好，请输出3个素数"
        )
    ]
)
print(response.content)
print("="*10)
print(response.content_blocks)

好的，以下是3个素数：2、3、5。
[{'type': 'reasoning', 'reasoning': '我们被要求输出3个素数。素数是指大于1的自然数，且除了1和它本身外没有其他因数。我们需要输出3个素数。可以是任意三个素数。简单输出如2,3,5。或者也可以是其他。由于没有指定格式，我们可以直接输出数字。'}, {'type': 'text', 'text': '好的，以下是3个素数：2、3、5。'}]


## 提示词模板

In [ ]:
#chat prompt template --推荐
from langchain_core.prompts import ChatPromptTemplate
cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个友好的ai助手,你的名字叫{name}'),
        ('user','你好'),
        ('assistant','你好'),
        ('user','{user_input}')
    ]
)
res = cpt.invoke({'name':'小智','user_input':'1+1=?'})
print(res)

messages=[SystemMessage(content='你是一个友好的ai助手,你的名字叫小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1+1=?', additional_kwargs={}, response_metadata={})]


#### 第二种方法


In [6]:
from langchain_core.prompts import ChatPromptTemplate
cpt = ChatPromptTemplate(
    [
        ('system','你是一个友好的ai助手,你的名字叫{name}'),
        ('user','你好'),
        ('assistant','你好'),
        ('user','{user_input}')
    ]
)
res = cpt.invoke({'name':'小智','user_input':'1+1=?'})
print(res)

messages=[SystemMessage(content='你是一个友好的ai助手,你的名字叫小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1+1=?', additional_kwargs={}, response_metadata={})]


#### 调用
1. invoke()
2. format()

In [1]:
from langchain_core.prompts import ChatPromptTemplate
cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个友好的ai助手,你的名字叫{name}'),
        ('user','你好'),
        ('assistant','你好'),
        ('user','{user_input}')
    ]
)
res = cpt.format(name='小美',user_input='2+2=?')
print(res)

System: 你是一个友好的ai助手,你的名字叫小美
Human: 你好
AI: 你好
Human: 2+2=?


### format_messages()

In [2]:
from langchain_core.prompts import ChatPromptTemplate
cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个友好的ai助手,你的名字叫{name}'),
        ('user','你好'),
        ('assistant','你好'),
        ('user','{user_input}')
    ]
)
res = cpt.format_messages(name='小美',user_input='2+2=?')
print(res)

[SystemMessage(content='你是一个友好的ai助手,你的名字叫小美', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='2+2=?', additional_kwargs={}, response_metadata={})]


### 融入实战

In [1]:
import base64
from langchain_deepseek import ChatDeepSeek
from langchain.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
class LLM:
    def __init__(self):
        load_dotenv(".env.local",override=True)
        self.api_key= os.getenv("DEEPSEEK_API_KEY").strip()
        self.model= os.getenv("MODEL").strip()
        self.url= os.getenv("DEEPSEEK_BASE_URL").strip()
        try:
            self.thinking_timeout= float(os.getenv("TINKING_TIMEOUT",30).strip())
            self.num_retries= int(os.getenv("RETRY_NUM",2).strip())
            self.temperature= int(os.getenv("TEMPERATURE",1).strip())
            self.stream= True if os.getenv("STREAM").strip() == 'True' else False
            self.max_token= int(os.getenv("MAX_TOKEN").strip()) if os.getenv("MAX_TOKEN").strip() else None
        except Exception as e:
            raise Exception(f'配置文件传入非法参数。错误信息：{e}')
        self.set_llm()
    def set_llm(self):
        self.llm = ChatDeepSeek(
            model= self.model,
            api_key= self.api_key,
            streaming= self.stream,
            api_base= self.url,
            temperature=self.temperature,
            request_timeout= self.thinking_timeout,
            max_tokens= self.max_token,
            max_retries= self.num_retries,
            extra_body={"thinking":{"type":"enabled"}}
            # model_kwargs=   {'tools':[]}##用来存放一些langchain没有列出但模型本身支持的，比如tools
            # configurable_fields= ('model','temperature') ## 用来允许 config中的configurable 覆盖
        )

cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个友好的ai助手,你的名字叫{name}'),
        ('user','你好'),
        ('assistant','你好'),
        ('user','{user_input}')
    ]
)
prompt = cpt.format(name='小美',user_input='2+2=?')

model = LLM().llm
response = model.invoke(
    prompt
)
print(response.content)
print("="*10)
print(response.content_blocks)

2+2=4
[{'type': 'reasoning', 'reasoning': '我们被问到："2+2=?" 这是一个简单的数学问题。作为AI，应该回答正确。答案：4。'}, {'type': 'text', 'text': '2+2=4'}]


In [5]:
from langchain_core.prompts import ChatPromptTemplate
cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个友好的ai助手,你的名字叫{name}'),
        ('user','你好'),
        ('assistant','你好'),
        ('user','{user_input}')
    ]
)
f_cpt = cpt.partial(name = '小帅')
res = f_cpt.format_messages(name = '小美',user_input='2+2=?')
print(res)

[SystemMessage(content='你是一个友好的ai助手,你的名字叫小美', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='2+2=?', additional_kwargs={}, response_metadata={})]


### 消息占位符

In [8]:
from langchain_core.prompts import ChatPromptTemplate

cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个名为{name}的人工智能助手'),
        ('placeholder','{conversation}')
    ]
)
res = cpt.format_messages(
    name = '小帅',
    conversation = [
        ('user','你好'),
        ('assistant','你好'),
        ('user','1+1 = ?')
    ]
)
print(res)


[SystemMessage(content='你是一个名为小帅的人工智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1+1 = ?', additional_kwargs={}, response_metadata={})]


In [9]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个名为{name}的人工智能助手'),
        MessagesPlaceholder(variable_name = 'conversation')
    ]
)
res = cpt.format_messages(
    name = '小帅',
    conversation = [
        ('user','你好'),
        ('assistant','你好'),
        ('user','1+1 = ?')
    ]
)
print(res)


[SystemMessage(content='你是一个名为小帅的人工智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1+1 = ?', additional_kwargs={}, response_metadata={})]


In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

cpt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个名为{name}的人工智能助手'),
        MessagesPlaceholder(variable_name = 'conversation')
    ]
)
res = cpt.format_messages(
    name = '小帅',
    conversation = [
        ('user','你好'),
        ('assistant','你好'),
        ('user','1+1 = ?')
    ]
)
print(res)
